In [2]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [3]:
from rag_helper import RAGBase
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [4]:
instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
    instructions=instructions,
)

In [5]:
answer = assistant.rag("How do I run Ollama locally?")
print(answer)

To run Ollama locally:

1. Install Ollama from https://ollama.com/download for your operating system.
   - macOS: download and install the `.pkg`
   - Windows: download and install the `.msi`
   - Linux: run:
     ```bash
     curl -fsSL https://ollama.com/install.sh | sh
     ```

2. Open a terminal and run:
   ```bash
   ollama run llama3
   ```
   This downloads the LLaMA 3 model, starts it locally, and opens a chat-like interface.

3. To check that the local server is running, use:
   ```bash
   curl http://localhost:11434
   ```
   You should see a response like:
   ```json
   {"models": [...]}
   ```

4. If you need to restart the Ollama server, run:
   ```bash
   !nohup ollama serve > nohup.out 2>&1 &
   ```


In [7]:
print(assistant.rag("How do I run Olama locally?"))

If you want to run it locally, you can do so if you’re comfortable setting up the needed tools yourself.

From the course FAQ:
- The course can be run locally instead of in Codespaces.
- You’ll need to set up Python, `uv`, Jupyter, Docker, and any other tools needed for the module.
- If you run locally, you should document your setup and keep your environment reproducible.

If you meant a specific tool named “Olama,” I don’t have any FAQ entry for that.


In [5]:
messages = [
    {"role": "user", "content": "I just discovered the course. Can I join it?"}
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
)

response.output_text

'Yes—usually you can still join, but it depends on the course’s enrollment rules and whether it’s still open.\n\nIf you want, I can help you figure it out quickly. Please share:\n- the course name or link\n- where it’s offered\n- whether you mean enrolling as a student, auditing, or joining late\n\nIf you’re asking what to send the instructor/admin, you can use:\n\n> Hi, I just discovered this course and I’m very interested in joining. Is it still possible to enroll at this point? If so, could you please let me know the next steps?\n\nIf you want, I can also help you write a more polite or more casual version.'

In [6]:
def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [7]:
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

In [8]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

response.output

[ResponseFunctionToolCall(arguments='{"query":"Can I join the course if I just discovered it? enrollment late join join course discovered"}', call_id='call_mViVvZXrjHgI0ETt9qCnQdVv', name='search', type='function_call', id='fc_01f75268331bda8c006a7a2333c9c0819a9c8c56313e7310a0', caller=None, namespace=None, status='completed')]

In [9]:
import json

call = response.output[0]
args = json.loads(call.arguments)

results = search(**args)
result_json = json.dumps(results, indent=2)

In [10]:
print(result_json)

[
  {
    "id": "74eb249bbf",
    "course": "llm-zoomcamp",
    "section": "General Course-Related Questions",
    "question": "I just discovered the course. Can I still join?",
    "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\u2019re still accepting submissions."
  },
  {
    "id": "5cc511f85b",
    "course": "llm-zoomcamp",
    "section": "General Course-Related Questions",
    "question": "Does the course certificate show the number of course hours?",
    "answer": "No. The certificate does not state a total number of hours."
  },
  {
    "id": "977bf7786c",
    "course": "llm-zoomcamp",
    "section": "General Course-Related Questions",
    "question": "Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?",
    "answer": "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked again

In [11]:
messages.extend(response.output)

messages.append({
    "type": "function_call_output",
    "call_id": call.call_id,
    "output": result_json,
})

In [12]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

print(response.output_text)

Yes — you can still join.

If you want a certificate, make sure you submit your project while submissions are still open.


In [13]:
usage = response.usage
usage.input_tokens, usage.output_tokens

(790, 29)

In [14]:
def calculate_gpt54mini_price(input_tokens, output_tokens):
    INPUT_PRICE_PER_MILLION = 0.15
    OUTPUT_PRICE_PER_MILLION = 0.60

    input_cost = (input_tokens / 1_000_000) * INPUT_PRICE_PER_MILLION
    output_cost = (output_tokens / 1_000_000) * OUTPUT_PRICE_PER_MILLION
    total_cost = input_cost + output_cost

    return {
        "input_cost": input_cost,
        "output_cost": output_cost,
        "total_cost": total_cost,
    }

result = calculate_gpt54mini_price(652, 33)
print("Total cost: $", round(result["total_cost"], 8))

Total cost: $ 0.0001176
